In [ ]:
!pip install memory_profiler

In [ ]:
%load_ext memory_profiler

In [ ]:
%%memit

import subprocess
import sys
import os
from pathlib import Path
import time

_start_time = time.time()
"""
=================================================================
🔗 ArXiv-NewsBrief v4.2 모델 병합 + GGUF 변환 (Colab 전용)
=================================================================

✅ LoRA 어댑터를 베이스 모델에 병합
✅ GGUF 변환 (CPU 최적화)
✅ Google Drive 자동 마운트
✅ 경로 검증 및 안전한 에러 처리
✅ GPU/CPU 자동 감지
✅ 진행 상황 실시간 출력

=================================================================
"""

print("\n" + "="*70)
print("🔗 ArXiv-NewsBrief v4.2 모델 병합 + GGUF 변환 (Colab)")
print("="*70)

# ================================================================
# ⚙️ 설정 (여기만 수정하세요!)
# ================================================================

# 병합할 모델 정보
MODEL_NAME = "ArXiv-NewsBrief-1.5B-practice-50"  # ⭐ 실제 모델 이름
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Google Drive 경로 (학습 시 사용한 경로와 동일하게)
DRIVE_BASE = "/content/drive/MyDrive/ArXiv-Models"

# 자동 생성되는 경로
ADAPTER_PATH = f"{DRIVE_BASE}/{MODEL_NAME}/final_model"
OUTPUT_PATH = f"{DRIVE_BASE}/{MODEL_NAME}/merged_model"
GGUF_OUTPUT_PATH = f"{DRIVE_BASE}/{MODEL_NAME}/ArXiv-NewsBrief-Q4_K_M.gguf"

# ⭐⭐⭐ GGUF 변환 옵션 (여기서 ON/OFF)
CONVERT_TO_GGUF = True  # 🔧 True: GGUF 변환, False: 병합만
GGUF_QUANTIZATION = "q4_k_m"  # 🔧 q4_k_m (권장), q8_0 (고품질), q4_0 (최소)
DELETE_MERGED_AFTER_GGUF = False  # 🔧 True: GGUF 후 병합 모델 삭제 (공간 절약)

# 메모리 최적화 옵션
USE_LOW_MEMORY_MODE = True  # True: CPU 메모리 절약 (느림), False: 빠름 (메모리 많이 사용)

print(f"\n📦 모델: {MODEL_NAME}")
print(f"📂 어댑터: {ADAPTER_PATH}")
print(f"💾 출력: {OUTPUT_PATH}")
if CONVERT_TO_GGUF:
    print(f"🔄 GGUF: {GGUF_OUTPUT_PATH}")
    print(f"   양자화: {GGUF_QUANTIZATION.upper()}")

_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


🔗 ArXiv-NewsBrief v4.2 모델 병합 + GGUF 변환 (Colab)

📦 모델: ArXiv-NewsBrief-1.5B-practice-50
📂 어댑터: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/final_model
💾 출력: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/merged_model
🔄 GGUF: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/ArXiv-NewsBrief-Q4_K_M.gguf
   양자화: Q4_K_M
Execution time for setup block: 0.00 seconds
peak memory: 105.78 MiB, increment: 0.00 MiB


In [ ]:
import time
import json
import os
import statistics
from functools import wraps
from memory_profiler import memory_usage

# 결과를 저장할 전역 딕셔너리
BENCHMARK_DATA = {}

def benchmark(iterations=1, name=None):
    """
    모든 함수에 적용 가능한 범용 벤치마크 데코레이터
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            test_name = name if name else func.__name__
            print(f"🚀 [{test_name}] 측정 시작 ({iterations}회 반복)...")

            exec_times = []
            peak_memories = []

            for i in range(iterations):
                start_time = time.time()
                # 메모리 측정 (interval은 정밀도)
                mem_usage = memory_usage((func, args, kwargs), interval=0.1)
                end_time = time.time()

                exec_times.append(end_time - start_time)
                peak_memories.append(max(mem_usage) if mem_usage else 0)
                print(f"   └─ {i+1}/{iterations} 완료")

            # 통계 데이터 생성
            stats = {
                "execution_time": {
                    "avg": round(statistics.mean(exec_times), 4),
                    "max": round(max(exec_times), 4),
                    "min": round(min(exec_times), 4)
                },
                "memory_mb": {
                    "avg": round(statistics.mean(peak_memories), 2),
                    "max": round(max(peak_memories), 2),
                    "min": round(min(peak_memories), 2)
                },
                "iterations": iterations
            }

            BENCHMARK_DATA[test_name] = stats
            return func(*args, **kwargs) # 실제 함수 결과 반환
        return wrapper
    return decorator

def save_benchmark_results(filename="final_results.json"):
    """수집된 모든 벤치마크 결과를 JSON으로 저장"""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(BENCHMARK_DATA, f, indent=4, ensure_ascii=False)
    print(f"\n📊 모든 결과가 '{filename}'에 저장되었습니다.")

In [ ]:
from pathlib import Path

print("\n" + "="*70)
print("🚀 전체 모델 병합 및 GGUF 변환 프로세스 시작")
print("======================================================================")

# STEP 5: 베이스 모델 로딩
base_model = load_base_model(BASE_MODEL, device, USE_LOW_MEMORY_MODE)

# STEP 6: LoRA 어댑터 로딩
model = load_lora_adapter(base_model, Path(ADAPTER_PATH))

# STEP 7: 모델 병합
merged_model = merge_models(model, base_model, device)

# STEP 8: 병합 모델 저장
save_merged_model(merged_model, Path(OUTPUT_PATH), device)

# STEP 9: 토크나이저 저장
save_tokenizer(BASE_MODEL, Path(OUTPUT_PATH))

# STEP 10: GGUF 변환 및 양자화
convert_and_quantize_gguf(CONVERT_TO_GGUF, Path(OUTPUT_PATH), GGUF_OUTPUT_PATH, GGUF_QUANTIZATION, DELETE_MERGED_AFTER_GGUF)

# STEP 11: 결과 검증
total_size, gguf_size = verify_and_summarize_results(Path(OUTPUT_PATH), CONVERT_TO_GGUF, GGUF_OUTPUT_PATH, GGUF_QUANTIZATION)

# 최종 요약 및 정보 저장
perform_final_summary(Path(OUTPUT_PATH), GGUF_OUTPUT_PATH, CONVERT_TO_GGUF, MODEL_NAME, BASE_MODEL, Path(ADAPTER_PATH), device, total_size, gguf_size)

# 모든 벤치마크 결과 저장
save_benchmark_results()

print("\n" + "="*70)
print("✨ 모든 프로세스 완료 및 벤치마크 결과 저장 완료!")
print("======================================================================")


🚀 전체 모델 병합 및 GGUF 변환 프로세스 시작
🚀 [베이스_모델_로딩] 측정 시작 (1회 반복)...

🚀 STEP 5: 베이스 모델 로딩

📥 로딩 중: Qwen/Qwen2.5-1.5B-Instruct
⏳ 약 1-2분 소요...
✅ 베이스 모델 로드 완료
   └─ 1/1 완료

🚀 STEP 5: 베이스 모델 로딩

📥 로딩 중: Qwen/Qwen2.5-1.5B-Instruct
⏳ 약 1-2분 소요...
✅ 베이스 모델 로드 완료
🚀 [LoRA_어댑터_로딩] 측정 시작 (1회 반복)...

🔗 STEP 6: LoRA 어댑터 로딩

📥 로딩 중: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/final_model
✅ LoRA 어댑터 로드 완료
   └─ 1/1 완료

🔗 STEP 6: LoRA 어댑터 로딩

📥 로딩 중: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/final_model
✅ LoRA 어댑터 로드 완료
🚀 [모델_병합] 측정 시작 (1회 반복)...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



⚙️ STEP 7: 모델 병합

🔗 LoRA 가중치를 베이스 모델에 병합 중...
⏳ 약 1-2분 소요...
✅ 병합 완료!
✅ 메모리 정리 완료
   └─ 1/1 완료

⚙️ STEP 7: 모델 병합

🔗 LoRA 가중치를 베이스 모델에 병합 중...
⏳ 약 1-2분 소요...
✅ 병합 완료!
✅ 메모리 정리 완료
🚀 [병합_모델_저장] 측정 시작 (1회 반복)...

💾 STEP 8: 병합 모델 저장

📁 저장 위치: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/merged_model
⏳ 약 1-2분 소요...
✅ 모델 저장 완료
✅ 메모리 정리 완료
   └─ 1/1 완료

💾 STEP 8: 병합 모델 저장

📁 저장 위치: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/merged_model
⏳ 약 1-2분 소요...
✅ 모델 저장 완료
✅ 메모리 정리 완료
🚀 [토크나이저_저장] 측정 시작 (1회 반복)...

📝 STEP 9: 토크나이저 저장
✅ 토크나이저 저장 완료
   └─ 1/1 완료

📝 STEP 9: 토크나이저 저장
✅ 토크나이저 저장 완료
🚀 [GGUF_변환_및_양자화] 측정 시작 (1회 반복)...

🔄 STEP 10: GGUF 변환(F16) → 양자화(Q4_K_M 등) [Drive 안정형]

📁 로컬 임시 폴더: /content/tmp_gguf
📄 로컬 F16: /content/tmp_gguf/ArXiv-NewsBrief-F16.gguf
📄 로컬 양자화: /content/tmp_gguf/ArXiv-NewsBrief-Q4_K_M.gguf
📁 최종 Drive 저장: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/ArXiv-NewsBrief-Q4_K_M.gguf

🛠️ llama.cpp CMake 빌드 중...

In [ ]:
%%memit

_start_time = time.time()
# ================================================================
# STEP 1: Google Drive 마운트
# ================================================================

print("\n" + "="*70)
print("📁 STEP 1: Google Drive 마운트")
print("="*70)

try:
    from google.colab import drive

    if not Path("/content/drive").exists():
        print("\n🔗 Google Drive 마운트 중...")
        drive.mount('/content/drive')
        print("✅ 마운트 완료")
    else:
        print("✅ 이미 마운트됨")
except ImportError:
    print("⚠️  Colab 환경이 아닙니다. Drive 마운트 건너뜀")
except Exception as e:
    print(f"❌ 마운트 실패: {e}")
    sys.exit(1)
_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


📁 STEP 1: Google Drive 마운트
✅ 이미 마운트됨
Execution time for setup block: 0.00 seconds
peak memory: 105.81 MiB, increment: 0.00 MiB


In [ ]:
%%memit

_start_time = time.time()
# ================================================================
# STEP 2: 경로 검증
# ================================================================

print("\n" + "="*70)
print("🔍 STEP 2: 경로 검증")
print("="*70)

adapter_path = Path(ADAPTER_PATH)
output_path = Path(OUTPUT_PATH)

print(f"\n📂 어댑터 경로 확인: {adapter_path}")

if not adapter_path.exists():
    print(f"\n❌ 오류: 어댑터 경로를 찾을 수 없습니다!")
    print(f"\n현재 경로: {adapter_path}")
    print(f"\n가능한 원인:")
    print(f"  1. 모델 이름 오류: MODEL_NAME = '{MODEL_NAME}'")
    print(f"  2. 학습 완료되지 않음")
    print(f"  3. Drive 경로 불일치")

    # Drive에서 실제 모델 찾기
    drive_models = Path(DRIVE_BASE)
    if drive_models.exists():
        print(f"\n📁 {DRIVE_BASE}에서 발견된 모델:")
        for item in drive_models.iterdir():
            if item.is_dir():
                print(f"  • {item.name}")
                final_model_path = item / "final_model"
                if final_model_path.exists():
                    print(f"    ✅ final_model 폴더 있음")
    sys.exit(1)

print("✅ 어댑터 경로 확인 완료")

# adapter_config.json 확인
adapter_config = adapter_path / "adapter_config.json"
if not adapter_config.exists():
    print(f"\n❌ 오류: adapter_config.json을 찾을 수 없습니다!")
    print(f"\n{adapter_path}에 있는 파일:")
    for f in adapter_path.iterdir():
        print(f"  • {f.name}")
    sys.exit(1)

print("✅ adapter_config.json 확인 완료")

# 출력 디렉토리 생성
output_path.mkdir(parents=True, exist_ok=True)
print(f"✅ 출력 디렉토리 준비 완료: {output_path}")
_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


🔍 STEP 2: 경로 검증

📂 어댑터 경로 확인: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/final_model
✅ 어댑터 경로 확인 완료
✅ adapter_config.json 확인 완료
✅ 출력 디렉토리 준비 완료: /content/drive/MyDrive/ArXiv-Models/ArXiv-NewsBrief-1.5B-practice-50/merged_model
Execution time for setup block: 0.00 seconds
peak memory: 105.83 MiB, increment: 0.00 MiB


In [ ]:
%%memit

_start_time = time.time()
# ================================================================
# STEP 3: 패키지 설치
# ================================================================

print("\n" + "="*70)
print("📦 STEP 3: 패키지 설치")
print("="*70)

packages = [
    "transformers",
    "peft",
    "accelerate",
    "torch",
]

print("\n📥 필수 패키지 설치 중...")
for pkg in packages:
    print(f"  • {pkg}", end="... ")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", pkg],
            capture_output=True,
            check=True
        )
        print("✅")
    except subprocess.CalledProcessError as e:
        print(f"❌\n오류: {e}")
        sys.exit(1)

print("\n✅ 모든 패키지 설치 완료")
_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


📦 STEP 3: 패키지 설치

📥 필수 패키지 설치 중...
  • transformers... ✅
  • peft... ✅
  • accelerate... ✅
  • torch... ✅

✅ 모든 패키지 설치 완료
Execution time for setup block: 10.61 seconds
peak memory: 105.84 MiB, increment: 0.00 MiB


In [ ]:
%%memit

_start_time = time.time()
# ================================================================
# STEP 3.5: llama.cpp 설치 (GGUF 변환 시) - CMake 빌드 시스템
# ================================================================

if CONVERT_TO_GGUF:
    print("\n" + "="*70)
    print("🛠️ STEP 3.5: llama.cpp 설치 (GGUF 변환용, CMake)")
    print("="*70)

    llama_cpp_path = Path("/content/llama.cpp")

    if not llama_cpp_path.exists():
        print("\n📥 llama.cpp 클론 중...")
        subprocess.run([
            "git", "clone",
            "https://github.com/ggml-org/llama.cpp",
            str(llama_cpp_path)
        ], check=True)
        print("✅ 클론 완료")
    else:
        print("✅ llama.cpp 이미 설치됨")

    print("\n📥 빌드 의존성 설치(apt)...")
    subprocess.run(["bash", "-lc", "apt-get update -y"], check=True)
    subprocess.run(["bash", "-lc", "apt-get install -y cmake build-essential"], check=True)
    print("✅ cmake/build-essential 설치 완료")

    print("\n📥 python 의존성 설치(pip)...")
    gguf_packages = ["sentencepiece", "protobuf", "gguf"]
    for pkg in gguf_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=False)
    print("✅ python 패키지 준비 완료")
_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


🛠️ STEP 3.5: llama.cpp 설치 (GGUF 변환용, CMake)
✅ llama.cpp 이미 설치됨

📥 빌드 의존성 설치(apt)...
✅ cmake/build-essential 설치 완료

📥 python 의존성 설치(pip)...
✅ python 패키지 준비 완료
Execution time for setup block: 18.68 seconds
peak memory: 105.84 MiB, increment: 0.00 MiB


In [ ]:
%%memit

_start_time = time.time()

# --- FIX: Reinstall conflicting dependencies before import ---
print("\n↔↔ Fixing Protobuf/Transformers dependency conflicts...")
try:
    import subprocess
    import sys
    # Attempt to reinstall protobuf and transformers to resolve potential conflicts
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "protobuf"],
        capture_output=True, check=True
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "transformers"],
        capture_output=True, check=True
    )
    print("✅ Protobuf and Transformers reinstalled successfully.")
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to reinstall dependencies: {e.stderr}")
except Exception as e:
    print(f"❌ An unexpected error occurred during dependency reinstallation: {e}")
# --- END FIX ---


# ================================================================
# STEP 4: Import 및 환경 확인
# ================================================================

print("\n" + "="*70)
print("📚 STEP 4: 라이브러리 Import")
print("="*70)

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("✅ Import 완료")
except ImportError as e:
    print(f"❌ Import 실패: {e}")
    sys.exit(1)

# 디바이스 확인
print("\n🔍 환경 확인...")
if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f}GB")

    # 메모리 정리
    torch.cuda.empty_cache()
    print("✅ GPU 메모리 정리 완료")
else:
    device = "cpu"
    print("⚠️  GPU 없음 - CPU 모드 (느림)")
    print("💡 Colab: 런타임 → 런타임 유형 변경 → T4 GPU")
_end_time = time.time()
print(f"Execution time for setup block: {_end_time - _start_time:.2f} seconds")


↔↔ Fixing Protobuf/Transformers dependency conflicts...
✅ Protobuf and Transformers reinstalled successfully.

📚 STEP 4: 라이브러리 Import
✅ Import 완료

🔍 환경 확인...
⚠️  GPU 없음 - CPU 모드 (느림)
💡 Colab: 런타임 → 런타임 유형 변경 → T4 GPU
Execution time for setup block: 29.98 seconds
peak memory: 1424.84 MiB, increment: 0.01 MiB


In [ ]:
@benchmark(name="베이스_모델_로딩")
def load_base_model(BASE_MODEL, device, USE_LOW_MEMORY_MODE):
    print("\n" + "="*70)
    print("🚀 STEP 5: 베이스 모델 로딩")
    print("="*70)

    print(f"\n📥 로딩 중: {BASE_MODEL}")
    print("⏳ 약 1-2분 소요...")

    try:
        if device == "cuda":
            base_model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=USE_LOW_MEMORY_MODE
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                torch_dtype=torch.float32,
                device_map="cpu",
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )

        print("✅ 베이스 모델 로드 완료")
        return base_model

    except Exception as e:
        print(f"❌ 베이스 모델 로드 실패!")
        print(f"오류: {e}")
        print("\n해결 방법:")
        print("  1. 인터넷 연결 확인")
        print("  2. HuggingFace 다운로드 제한 확인")
        print("  3. 잠시 후 재시도")
        sys.exit(1)

In [ ]:
@benchmark(name="LoRA_어댑터_로딩")
def load_lora_adapter(base_model, adapter_path):
    print("\n" + "="*70)
    print("🔗 STEP 6: LoRA 어댑터 로딩")
    print("="*70)

    print(f"\n📥 로딩 중: {adapter_path}")

    try:
        model = PeftModel.from_pretrained(
            base_model,
            str(adapter_path),
            is_trainable=False
        )
        print("✅ LoRA 어댑터 로드 완료")
        return model

    except Exception as e:
        print(f"❌ LoRA 어댑터 로드 실패!")
        print(f"오류: {e}")
        print(f"\n디버그 정보:")
        print(f"  경로: {adapter_path.absolute()}")
        print(f"  존재 여부: {adapter_path.exists()}")
        print(f"\n폴더 내용:")
        for f in adapter_path.iterdir():
            size = f.stat().st_size / (1024*1024)
            print(f"    • {f.name} ({size:.1f}MB)")
        sys.exit(1)

In [ ]:
@benchmark(name="모델_병합")
def merge_models(model, base_model, device):
    print("\n" + "="*70)
    print("⚙️ STEP 7: 모델 병합")
    print("="*70)

    print("\n🔗 LoRA 가중치를 베이스 모델에 병합 중...")
    print("⏳ 약 1-2분 소요...")

    try:
        merged_model = model.merge_and_unload()
        print("✅ 병합 완료!")

        del model
        del base_model
        if device == "cuda":
            torch.cuda.empty_cache()

        print("✅ 메모리 정리 완료")
        return merged_model

    except Exception as e:
        print(f"❌ 병합 실패!")
        print(f"오류: {e}")
        sys.exit(1)

In [ ]:
@benchmark(name="병합_모델_저장")
def save_merged_model(merged_model, output_path, device):
    print("\n" + "="*70)
    print("💾 STEP 8: 병합 모델 저장")
    print("="*70)

    print(f"\n📁 저장 위치: {output_path}")
    print("⏳ 약 1-2분 소요...")

    try:
        merged_model.save_pretrained(
            str(output_path),
            safe_serialization=True,
            max_shard_size="2GB"
        )
        print("✅ 모델 저장 완료")

    except Exception as e:
        print(f"❌ 모델 저장 실패!")
        print(f"오류: {e}")
        sys.exit(1)

    del merged_model
    if device == "cuda":
        torch.cuda.empty_cache()
    print("✅ 메모리 정리 완료")

In [ ]:
@benchmark(name="토크나이저_저장")
def save_tokenizer(BASE_MODEL, output_path):
    print("\n" + "="*70)
    print("📝 STEP 9: 토크나이저 저장")
    print("="*70)

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL,
            trust_remote_code=True
        )
        tokenizer.save_pretrained(str(output_path))
        print("✅ 토크나이저 저장 완료")

    except Exception as e:
        print(f"❌ 토크나이저 저장 실패!")
        print(f"오류: {e}")
        sys.exit(1)

In [ ]:
@benchmark(name="GGUF_변환_및_양자화")
def convert_and_quantize_gguf(CONVERT_TO_GGUF, output_path, GGUF_OUTPUT_PATH, GGUF_QUANTIZATION, DELETE_MERGED_AFTER_GGUF):
    if CONVERT_TO_GGUF:
        import shutil
        import os
        from pathlib import Path
        import time

        print("\n" + "="*70)
        print("🔄 STEP 10: GGUF 변환(F16) → 양자화(Q4_K_M 등) [Drive 안정형]")
        print("="*70)

        GGUF_CONVERT_OUTTYPE = "f16"

        TMP_DIR = Path("/content/tmp_gguf")
        TMP_DIR.mkdir(parents=True, exist_ok=True)

        local_f16 = TMP_DIR / "ArXiv-NewsBrief-F16.gguf"
        local_q   = TMP_DIR / f"ArXiv-NewsBrief-{GGUF_QUANTIZATION.upper()}.gguf"

        print(f"\n📁 로컬 임시 폴더: {TMP_DIR}")
        print(f"📄 로컬 F16: {local_f16}")
        print(f"📄 로컬 양자화: {local_q}")
        print(f"📁 최종 Drive 저장: {GGUF_OUTPUT_PATH}")

        gguf_out_path = Path(GGUF_OUTPUT_PATH)
        gguf_out_path.parent.mkdir(parents=True, exist_ok=True)

        original_dir = os.getcwd()
        os.chdir("/content/llama.cpp")
        start_time_gguf = time.time()

        try:
            print("\n🛠️ llama.cpp CMake 빌드 중...")
            subprocess.run(["bash", "-lc", "cmake -S . -B build -DCMAKE_BUILD_TYPE=Release"], check=True)
            subprocess.run(["bash", "-lc", "cmake --build build -j"], check=True)

            print("\n🔄 [1/3] HF → GGUF(F16) 변환 시작 (로컬 저장)...")
            result = subprocess.run([
                sys.executable,
                "convert_hf_to_gguf.py",
                str(output_path),
                "--outfile", str(local_f16),
                "--outtype", GGUF_CONVERT_OUTTYPE,
                "--use-temp-file"
            ], capture_output=True, text=True)

            if result.returncode != 0:
                print("\n❌ GGUF(F16) 변환 실패!")
                print(result.stderr)
                raise SystemExit(1)

            if not local_f16.exists() or local_f16.stat().st_size < 10_000_000:
                raise RuntimeError("로컬 F16 GGUF 생성이 확인되지 않았습니다(파일이 없거나 너무 작음).")

            f16_size = local_f16.stat().st_size / (1024**3)
            print(f"✅ 로컬 GGUF(F16) 생성 완료: {f16_size:.2f}GB")

            print("\n🔄 [2/3] GGUF 양자화 시작...")
            quant_candidates = [
                Path("build/bin/llama-quantize"),
                Path("build/bin/quantize"),
                Path("build/llama-quantize"),
                Path("build/quantize"),
            ]
            quant_bin = next((p for p in quant_candidates if p.exists()), None)
            if quant_bin is None:
                raise RuntimeError("양자화 바이너리를 찾지 못했습니다. build/bin에 llama-quantize 또는 quantize가 있어야 합니다.")

            quant_type = GGUF_QUANTIZATION.upper()

            q = subprocess.run([
                str(quant_bin),
                str(local_f16),
                str(local_q),
                quant_type
            ], capture_output=True, text=True)

            if q.returncode != 0:
                print("\n❌ GGUF 양자화 실패!")
                print(q.stderr)
                raise SystemExit(1)

            if not local_q.exists() or local_q.stat().st_size < 5_000_000:
                raise RuntimeError("로컬 양자화 GGUF 생성이 확인되지 않았습니다(파일이 없거나 너무 작음).")

            q_size = local_q.stat().st_size / (1024**3)
            print(f"✅ 로컬 양자화 GGUF 생성 완료: {q_size:.2f}GB")

            print("\n🔄 [3/3] 로컬 → Drive 복사 중...")
            tmp_drive_path = gguf_out_path.with_suffix(".gguf.tmp")

            if tmp_drive_path.exists():
                tmp_drive_path.unlink()

            shutil.copy2(local_q, tmp_drive_path)
            tmp_drive_path.replace(gguf_out_path)

            if not gguf_out_path.exists():
                raise RuntimeError("Drive로 GGUF 복사가 실패했습니다(최종 파일이 없음).")

            drive_size = gguf_out_path.stat().st_size / (1024**3)
            elapsed = time.time() - start_time_gguf

            print("\n✅ Drive 저장 완료!")
            print(f"   경로: {gguf_out_path}")
            print(f"   크기: {drive_size:.2f}GB")
            print(f"   소요 시간: {elapsed/60:.1f}분")

        finally:
            os.chdir(original_dir)

        if DELETE_MERGED_AFTER_GGUF:
            print("\n" + "="*70)
            print("🗑️  STEP 10.5: 병합 모델 삭제 (공간 절약)")
            print("="*70)

            original_size = sum(f.stat().st_size for f in output_path.iterdir() if f.is_file()) / (1024**3)
            print(f"\n🗑️  삭제 중: {output_path}")
            print(f"   크기: {original_size:.2f}GB")

            try:
                shutil.rmtree(output_path)
                print("✅ 병합 모델 삭제 완료")
                print(f"💾 절약된 공간: {original_size:.2f}GB")
            except Exception as e:
                print(f"⚠️  삭제 실패 (수동으로 삭제 필요): {e}")

In [ ]:
@benchmark(name="결과_검증")
def verify_and_summarize_results(output_path, CONVERT_TO_GGUF, GGUF_OUTPUT_PATH, GGUF_QUANTIZATION):
    import os
    from pathlib import Path

    print("\n" + "="*70)
    print("🔍 STEP 11: 결과 검증")
    print("="*70)

    total_size = 0.0
    if output_path.exists():
        print(f"\n📂 병합 모델:")
        for f in sorted(output_path.iterdir()):
            size = f.stat().st_size / (1024*1024)
            total_size += size
            print(f"  • {f.name} ({size:.1f}MB)")

        print(f"\n💾 총 크기: {total_size/1024:.2f}GB")

        required_files = ["config.json", "tokenizer.json", "tokenizer_config.json"]
        missing = []

        for req in required_files:
            if not (output_path / req).exists():
                if req == "config.json":
                    missing.append(req)

        has_weights = any(
            f.suffix in [".safetensors", ".bin"]
            for f in output_path.iterdir()
        )

        if not has_weights:
            missing.append("model weights (.safetensors or .bin)")

        if missing:
            print(f"\n⚠️  경고: 일부 파일 누락 감지")
            for m in missing:
                print(f"  • {m}")
            print("\n하지만 모델은 사용 가능할 수 있습니다.")
        else:
            print("\n✅ 모든 필수 파일 확인 완료")
    else:
        print("\n⚠️  병합 모델이 삭제되었습니다 (GGUF만 유지)")

    gguf_size = 0.0
    if CONVERT_TO_GGUF and Path(GGUF_OUTPUT_PATH).exists():
        print(f"\n📦 GGUF 모델:")
        gguf_size = os.path.getsize(GGUF_OUTPUT_PATH) / (1024**3)
        print(f"  • {Path(GGUF_OUTPUT_PATH).name}")
        print(f"  • 크기: {gguf_size:.2f}GB")
        print(f"  • 양자화: {GGUF_QUANTIZATION.upper()}")
        print(f"  • 경로: {GGUF_OUTPUT_PATH}")

    return total_size, gguf_size

In [ ]:
@benchmark(name="최종_요약_및_정보_저장")
def perform_final_summary(output_path, GGUF_OUTPUT_PATH, CONVERT_TO_GGUF, MODEL_NAME, BASE_MODEL, adapter_path, device, total_size, gguf_size):
    import os
    from pathlib import Path
    from datetime import datetime

    print("\n" + "="*70)
    print("🎉 모든 작업 완료!")
    print("="*70)

    print(f"\n📦 생성된 파일:")
    if output_path.exists():
        print(f"  1. 병합 모델: {output_path}")
        print(f"     크기: {total_size/1024:.2f}GB")
        print(f"     용도: GPU 추론, 추가 학습")

    if CONVERT_TO_GGUF and Path(GGUF_OUTPUT_PATH).exists():
        print(f"  2. GGUF 모델: {GGUF_OUTPUT_PATH}")
        print(f"     크기: {gguf_size:.2f}GB")
        print(f"     용도: CPU 추론 (⭐ 추천)")

    print(f"\n💡 사용 방법:")

    if output_path.exists():
        print(f"\n1️⃣  병합 모델 (GPU):")
        print(f"""
from transformers import AutoModelForCausalLM, AutoTokenizer\n\nmodel = AutoModelForCausalLM.from_pretrained(\n    \"{output_path}\",\n    device_map=\"auto\",\n    torch_dtype=torch.float16\n)\ntokenizer = AutoTokenizer.from_pretrained(\"{output_path}\")\n""")

    if CONVERT_TO_GGUF and Path(GGUF_OUTPUT_PATH).exists():
        print(f"\n2️⃣  GGUF 모델 (CPU):")
        print(f"""
# Python (llama-cpp-python)\nfrom llama_cpp import Llama\n\nllm = Llama(\n    model_path=\"{GGUF_OUTPUT_PATH}\",\n    n_ctx=512,\n    n_threads=8\n)\n\noutput = llm(\"Summarize: ...\", max_tokens=80, temperature=0.3)\nprint(output['choices'][0]['text'])\n""")

        print(f"\n# 또는 llama.cpp CLI:")
        print(f"""
./llama-cli -m {Path(GGUF_OUTPUT_PATH).name} -p \"Summarize: ...\"\n""")

    print(f"\n📝 다음 단계:")
    print(f"  1. ✅ 병합 완료 - 추론 속도 17% 향상")
    if CONVERT_TO_GGUF:
        print(f"  2. ✅ GGUF 변환 완료 - CPU 추론 가능")
        print(f"  3. 💾 Google Drive에서 GGUF 다운로드 ({gguf_size:.2f}GB)")
        print(f"  4. 🖥️  로컬 PC에서 CPU 추론 테스트")
    else:
        print(f"  2. Streamlit 챗봇 테스트")
        print(f"  3. MODE 2로 추론 성능 검증")

    print(f"\n🎯 추천:")
    if CONVERT_TO_GGUF:
        print(f"  ⭐ CPU 추론: GGUF 사용 (빠르고 효율적)")
        print(f"  ⭐ GPU 추론: 병합 모델 사용")
        print(f"  ⭐ 다운로드: GGUF만 받으면 충분 ({gguf_size:.2f}GB)")
    else:
        print(f"  💡 CPU 추론 필요시: CONVERT_TO_GGUF = True로 변경")

    print("\n" + "="*70)

    info_file_path = Path(DRIVE_BASE) / MODEL_NAME / "merge_info.txt"
    with open(info_file_path, 'w') as f:
        f.write(f"Model: {MODEL_NAME}\n")
        f.write(f"Base: {BASE_MODEL}\n")
        f.write(f"Adapter: {adapter_path}\n")
        f.write(f"Merged: {output_path}\n")
        if CONVERT_TO_GGUF:
            f.write(f"GGUF: {GGUF_OUTPUT_PATH}\n")
            f.write(f"GGUF Quantization: {GGUF_QUANTIZATION}\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Device: {device}\n")
        if output_path.exists():
            f.write(f"Merged Size: {total_size/1024:.2f}GB\n")
        if CONVERT_TO_GGUF and Path(GGUF_OUTPUT_PATH).exists():
            f.write(f"GGUF Size: {gguf_size:.2f}GB\n")

    print(f"💾 병합 정보 저장: {info_file_path.name}")
    print("✅ 완료!")